# 实验2：词向量的学习与分析

> **课程**：自然语言处理  
> **实验对象**：本科生  
> **实验主题**：从独热表示到分布式表示，理解并实现 Word2Vec 的基本思想  
> **建议学时**：2-3 学时  
> **实验方式**：上机独立实践

---

# 一、实验目标

完成本实验后，学生应能够：

1. 区分独热表示、词袋表示和分布式表示。
2. 理解词向量能够通过向量距离表达词语之间的语义或用法相似性。
3. 理解“词的含义由上下文决定”的分布式假设。
4. 能够根据文本和窗口大小构造 Word2Vec 训练样本。
5. 理解 CBOW 与 Skip-gram 的输入、输出和训练目标。
6. 使用 PyTorch 实现一个小型 Skip-gram 词向量模型。
7. 使用余弦相似度寻找相近词，并解释实验结果。
8. 了解负采样、层次 Softmax、GloVe 与 ELMo 的基本区别。
9. 能够分析小语料训练词向量的局限性。

---

# 二、实验任务与学时安排

| 任务 | 内容 | 类型 | 建议时间 |
|---|---|---|---:|
| 任务1 | 独热表示与分布式表示对比 | 必做 | 15 min |
| 任务2 | 构造上下文窗口和训练样本 | 必做 | 20 min |
| 任务3 | 实现并训练 Skip-gram | 必做 | 40-50 min |
| 任务4 | 词向量相似度分析 | 必做 | 20 min |
| 任务5 | 可视化或参数对比 | 选做 | 15 min |
| 任务6 | GloVe/ELMo 对比讨论 | 必做书面分析 | 20 min |

## 最低完成标准

学生至少完成：

- 独热表示和分布式表示对比；
- 训练样本构造；
- Skip-gram 模型训练；
- 相似词查询；
- 实验分析和报告提交。

---

# 三、理论导入：为什么需要词向量

## 1. 独热表示的问题

假设词表为：

```text
[猫, 狗, 鸟, 鱼]
```

则：

```text
猫 → [1, 0, 0, 0]
狗 → [0, 1, 0, 0]
鸟 → [0, 0, 1, 0]
鱼 → [0, 0, 0, 1]
```

这种表示有两个明显问题：

1. 向量维度等于词表大小，词表越大，向量越稀疏；
2. 任意两个不同词语之间的距离都相同，不能表达“猫”和“狗”可能比“猫”和“飞机”更相似。

## 2. 分布式表示

分布式表示使用低维、稠密、连续的实数向量表示词语。例如：

```text
猫 → [0.21, 0.74, -0.11]
狗 → [0.18, 0.70, -0.08]
飞机 → [-0.63, 0.12, 0.91]
```

上面仅为说明格式而人工给出的示意数值，不作为实验结果。任务1将从语料统计中计算低维表示，任务3将通过训练得到 Skip-gram 词向量。

> 核心思想：出现在相似上下文中的词，往往具有相似的语义或用法。

---

# 四、任务1：独热表示与分布式表示对比

**任务产出：**同一组词的两种向量、完整词对指标表、相似度矩阵及书面分析。建议20—25分钟。

本任务先用可检查的教学语料构建共现矩阵，再用教师提供的 SVD 代码压缩为3维稠密表示。学生不必推导 SVD；重点观察“上下文统计如何变成向量”。这是统计式低维词表示演示，**不是 Word2Vec，也不是 GloVe 的实现**。任务3训练 Skip-gram 后还要再次对比。

## 1. 同一词表的独热表示



In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations

demo_words = ["猫", "狗", "鸟", "飞机"]
demo_ids = {w: i for i, w in enumerate(demo_words)}
demo_onehot = np.eye(len(demo_words), dtype=float)
print(pd.DataFrame(demo_onehot, index=demo_words, columns=demo_words))



## 2. 从语料统计上下文

以下是**为观察现象而设计的短语语料**，不是自然语言分布的代表样本。保留每条短语边界，在左右各2个词的窗口内，统计四个目标词周围出现的上下文词。



In [ ]:
demo_sentences = [
    "可爱 猫 宠物", "可爱 狗 宠物",
    "家中 猫 陪伴", "家中 狗 陪伴",
    "猫 吃 鱼", "狗 吃 肉",
    "鸟 天空 飞翔", "鸟 树林 飞翔",
    "飞机 天空 飞行", "飞机 机场 飞行",
    "鸟 空中 移动", "飞机 空中 移动",
]
demo_tokens = [s.split() for s in demo_sentences]
demo_contexts = sorted({w for sent in demo_tokens for w in sent} - set(demo_words))
demo_context_ids = {w: i for i, w in enumerate(demo_contexts)}
demo_counts = np.zeros((len(demo_words), len(demo_contexts)), dtype=float)
for sent in demo_tokens:
    for pos, word in enumerate(sent):
        if word not in demo_ids:
            continue
        for j in range(max(0, pos - 2), min(len(sent), pos + 3)):
            if j != pos and sent[j] in demo_context_ids:
                demo_counts[demo_ids[word], demo_context_ids[sent[j]]] += 1
print(pd.DataFrame(demo_counts, index=demo_words, columns=demo_contexts).to_string())



**手工检查：**“猫—可爱”的计数是多少？“猫—宠物”与“狗—宠物”的计数是否相同？共现矩阵仍然不是本任务最终的低维表示。

## 3. 得到3维分布式表示

先按行归一化计数，再进行截断 SVD，保留3个方向，取 `U[:, :3] * S[:3]` 作为词向量。NumPy 的 SVD 返回满足 `X = U @ diag(S) @ Vh` 的分解；这里只提供调用，不要求学生实现分解算法。



In [ ]:
demo_x = demo_counts / np.maximum(np.linalg.norm(demo_counts, axis=1, keepdims=True), 1e-12)
demo_u, demo_s, demo_vh = np.linalg.svd(demo_x, full_matrices=False)
demo_dense = demo_u[:, :3] * demo_s[:3]
assert demo_dense.shape == (4, 3)
assert np.all(np.linalg.norm(demo_dense, axis=1) > 0)
print(pd.DataFrame(demo_dense, index=demo_words, columns=["维度1", "维度2", "维度3"]).round(4))
print("独热表示形状：", demo_onehot.shape)
print("分布式表示形状：", demo_dense.shape)



这里的数值来自上述语料计算，不是人为指定“猫和狗应该相似”。但语料本身经过教学设计，因此不能把结果当作通用语义能力的证据。向量各维不要擅自命名为“动物性”“飞行性”等语义属性。

## 4. 用相同词对、相同指标生成对照表



In [ ]:
def compare_representations(words, onehot_vectors, dense_vectors):
    records = []
    for i, j in combinations(range(len(words)), 2):
        row = {"词对": f"{words[i]}—{words[j]}"}
        for label, vectors in [("独热", onehot_vectors), ("分布式", dense_vectors)]:
            a, b = vectors[i], vectors[j]
            denominator = np.linalg.norm(a) * np.linalg.norm(b)
            row[f"{label}_欧氏距离"] = float(np.linalg.norm(a - b))
            row[f"{label}_余弦相似度"] = float(a @ b / denominator) if denominator > 0 else np.nan
        records.append(row)
    return pd.DataFrame(records)

demo_comparison = compare_representations(demo_words, demo_onehot, demo_dense)
print(demo_comparison.round(4).to_string(index=False))
assert np.allclose(demo_comparison["独热_欧氏距离"], np.sqrt(2))
assert np.allclose(demo_comparison["独热_余弦相似度"], 0)



共比较6组不同词对。请先核对独热列，再观察分布式列中的“猫—狗”“鸟—飞机”“猫—飞机”。比较重点是各表示内部能否区分词对，不要直接把两个空间的原始欧氏距离大小解释为优劣。

## 5. 显示相似度矩阵并保存结果



In [ ]:
from pathlib import Path

def row_cosine_matrix(vectors):
    normalized = vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)
    return normalized @ normalized.T

print("独热表示的余弦相似度矩阵：")
print(pd.DataFrame(row_cosine_matrix(demo_onehot), index=demo_words, columns=demo_words).round(4))
print("分布式表示的余弦相似度矩阵：")
print(pd.DataFrame(row_cosine_matrix(demo_dense), index=demo_words, columns=demo_words).round(4))
Path("output").mkdir(exist_ok=True)
demo_comparison.to_csv("output/task1_comparison.csv", index=False, encoding="utf-8-sig")



## 6. 学生分析与验收

请在下面填写分析，不要只提交截图：

1. 独热表示和分布式表示的维度分别是多少？
2. 为什么独热表示对所有不同词对给出同样的结果？请引用表中的数值。
3. 分布式表示中，哪组词更接近？从原始短语中找出共享上下文作为解释。
4. “鸟”和“飞机”共享一些上下文，是否意味着它们是同义词？
5. 将一条关于“狗”的短语改为飞行相关短语，重新运行步骤2—5，记录“猫—狗”的相似度变化。不得直接修改向量数值。
6. 本任务只说明了什么？为什么还不能证明词向量在真实应用中可靠？

**验收材料：**共现矩阵、两种词向量、6组词对对照表、修改语料前后的结果及解释。

**技术参考：**NumPy 官方文档 `numpy.linalg.svd`

---

# 五、任务2：构造上下文窗口与训练样本

## 1. 准备小型语料

本实验使用一个规模较小、便于课堂运行的示例语料。语料内容专门设计为包含若干可观察的上下文关系。



In [ ]:
sentences = [
    "猫 喜欢 吃 鱼",
    "狗 喜欢 吃 肉",
    "猫 和 狗 都 是 宠物",
    "小猫 是 可爱 的 宠物",
    "小狗 是 忠诚 的 宠物",
    "鱼 生活 在 水中",
    "鸟 喜欢 在 天空 飞翔",
    "飞机 在 天空 飞行",
]

corpus = " ".join(sentences).split()
print("语料词数：", len(corpus))
print("前20个词：", corpus[:20])



> 这里使用空格分隔词语，是为了把实验重点放在词向量学习上，而不是中文分词。真实中文语料需要先进行分词。

## 2. 建立词表



In [ ]:
from collections import Counter

word_counts = Counter(corpus)
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

print("词表大小：", len(vocab))
print("词表：", vocab)



## 3. 构造 Skip-gram 训练样本

Skip-gram 的基本任务是：给定中心词，预测窗口内的上下文词。

例如，句子：

```text
猫 喜欢 吃 鱼
```

窗口大小为 1 时，可以构造：

```text
(猫, 喜欢)
(喜欢, 猫)
(喜欢, 吃)
(吃, 喜欢)
(吃, 鱼)
(鱼, 吃)
```



In [ ]:
def make_skipgram_pairs(tokens, word_to_id, window_size=1):
    pairs = []
    for center_pos, center_word in enumerate(tokens):
        center_id = word_to_id[center_word]
        left = max(0, center_pos - window_size)
        right = min(len(tokens), center_pos + window_size + 1)

        for context_pos in range(left, right):
            if context_pos == center_pos:
                continue
            context_word = tokens[context_pos]
            pairs.append((center_id, word_to_id[context_word]))
    return pairs

sentence_tokens = [sentence.split() for sentence in sentences]
pairs = [pair for tokens in sentence_tokens
         for pair in make_skipgram_pairs(tokens, word_to_id, window_size=1)]

print("训练样本数：", len(pairs))
for center_id, context_id in pairs[:10]:
    print(id_to_word[center_id], "→", id_to_word[context_id])



## 4. 比较不同窗口大小



In [ ]:
for window_size in [1, 2, 3]:
    current_pairs = [pair for tokens in sentence_tokens
                     for pair in make_skipgram_pairs(tokens, word_to_id, window_size)]
    print(f"窗口大小={window_size}, 样本数={len(current_pairs)}")



## 思考题

1. 窗口增大后，训练样本数量如何变化？
2. 小窗口和大窗口分别更容易学习哪类信息？
3. 如果窗口跨越句子边界，会产生什么问题？如何改进？

---

# 六、任务3：实现 Skip-gram 词向量模型

## 1. 模型思想

Skip-gram 可以理解为一个分类任务：

```text
中心词 → 上下文词
```

模型包含一个词向量矩阵。输入中心词的编号后，通过查表获得其向量，再使用线性层预测上下文词。

本实验采用完整 Softmax，便于理解模型结构。真实的大词表训练通常会使用负采样或层次 Softmax 以降低计算成本。

## 2. 导入依赖



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)



## 3. 定义模型



In [ ]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size, embedding_dim=10):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, center_words):
        vectors = self.embedding(center_words)
        logits = self.output(vectors)
        return logits



## 4. 准备训练数据



In [ ]:
centers = torch.tensor([p[0] for p in pairs], dtype=torch.long)
contexts = torch.tensor([p[1] for p in pairs], dtype=torch.long)

print("中心词张量形状：", centers.shape)
print("上下文词张量形状：", contexts.shape)



## 5. 训练模型



In [ ]:
embedding_dim = 10
model = SkipGram(len(vocab), embedding_dim=embedding_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.03)

loss_history = []

for epoch in range(1000):
    optimizer.zero_grad()
    logits = model(centers)
    loss = criterion(logits, contexts)
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if (epoch + 1) % 200 == 0:
        print(f"epoch={epoch + 1}, loss={loss.item():.4f}")



## 6. 绘制损失曲线



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Skip-gram Training Loss")
plt.grid(alpha=0.3)
plt.show()



## 结果检查

通常情况下，随着训练进行，损失会总体下降。由于语料规模很小、训练样本存在重复，损失曲线可能会有波动。

> 注意：本实验采用的是教学用小模型。损失下降并不等于学到了可靠的通用语义，只能说明模型在当前小语料上拟合了部分上下文关系。

---

# 七、任务4：词向量相似度分析

## 1. 提取词向量



In [ ]:
embeddings = model.embedding.weight.detach()
print("词向量矩阵形状：", embeddings.shape)



## 2. 计算余弦相似度



In [ ]:
import torch.nn.functional as F

def cosine_similarity(word_a, word_b):
    vec_a = embeddings[word_to_id[word_a]].unsqueeze(0)
    vec_b = embeddings[word_to_id[word_b]].unsqueeze(0)
    return F.cosine_similarity(vec_a, vec_b).item()

for pair in [("猫", "狗"), ("天空", "飞机"), ("猫", "飞机"), ("鱼", "水中")]:
    print(pair, cosine_similarity(*pair))



## 3. 查询最相似词



In [ ]:
def most_similar(word, top_k=5):
    target_id = word_to_id[word]
    target = embeddings[target_id].unsqueeze(0)
    similarities = F.cosine_similarity(target, embeddings)

    result = []
    for idx in torch.argsort(similarities, descending=True):
        idx = idx.item()
        if idx == target_id:
            continue
        result.append((id_to_word[idx], float(similarities[idx])))
        if len(result) >= top_k:
            break
    return result

for word in ["猫", "狗", "天空", "宠物"]:
    print(f"{word} 的相似词：", most_similar(word))



## 结果分析要求

请在实验报告中回答：

1. 哪些词语的相似度较高？是否符合语料中的上下文关系？
2. “猫”和“狗”是否比“猫”和“飞机”更相似？为什么？
3. 如果实验结果不符合预期，可能有哪些原因？
4. 为什么不能只根据一次训练结果就断言两个词在真实语言中一定相似？

---

## 4. 回到任务1：对比真正训练得到的 Skip-gram 向量

这一步使用任务3训练后的 `embeddings`，不是任务1的 SVD 向量。对同样四个词，重新输出独热与训练所得表示的指标表。任务1的函数 `compare_representations` 需要已执行。



In [ ]:
trained_compare_words = ["猫", "狗", "鸟", "飞机"]
trained_ids = [word_to_id[w] for w in trained_compare_words]
trained_vectors = embeddings[trained_ids].cpu().numpy()
trained_onehot = np.eye(len(vocab), dtype=float)[trained_ids]
trained_comparison = compare_representations(trained_compare_words, trained_onehot, trained_vectors)
print(trained_comparison.round(4).to_string(index=False))
trained_comparison.to_csv("output/task4_trained_comparison.csv", index=False, encoding="utf-8-sig")



记录各词对的结果并与任务1比较。两次实验的语料、目标和维度不同，不能据此判断 SVD 或 Skip-gram 谁更好。小语料上的结果不保证符合直觉；分析过程正确比得到预设的相似词排名更重要。

---

# 八、任务5：词向量二维可视化（选做）

使用 PCA 将高维词向量投影到二维空间。



In [ ]:
from sklearn.decomposition import PCA

vectors = embeddings.numpy()
pca = PCA(n_components=2)
points = pca.fit_transform(vectors)

plt.figure(figsize=(8, 6))
plt.scatter(points[:, 0], points[:, 1])

for i, word in enumerate(vocab):
    plt.text(points[i, 0] + 0.01, points[i, 1] + 0.01, word)

plt.title("Word Embeddings Visualized by PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(alpha=0.3)
plt.show()



## 可视化分析

- 观察“宠物”“猫”“狗”等词是否出现相对接近的区域；
- 观察“天空”“飞机”“飞行”等词是否形成局部聚集；
- 说明二维图只能作为直观观察，不能完整代表原始空间中的距离关系。

---

# 九、任务6：Word2Vec、GloVe 与 ELMo 对比

请结合课堂内容完成下表。

| 方法 | 表示类型 | 主要信息来源 | 是否静态 | 主要特点 |
|---|---|---|---|---|
| 独热表示 | 稀疏离散 | 词表编号 | 是 | 维度高，无法表达语义相似性 |
| Word2Vec | 稠密分布式 | 局部上下文 | 是 | 通过预测任务学习词向量 |
| GloVe | 稠密分布式 | 全局共现统计 | 是 | 利用词-上下文共现矩阵 |
| ELMo | 上下文相关表示 | 双向语言模型 | 否 | 同一个词在不同语境中可以有不同表示 |

## 思考题

1. Word2Vec 的 CBOW 和 Skip-gram 分别是什么任务？
2. Word2Vec 使用局部上下文，GloVe 使用全局共现信息，这两种信息有什么差异？
3. 为什么 ELMo 能够缓解一词多义问题，而传统 Word2Vec 不能充分解决？
4. 什么是静态词向量？什么是上下文相关词向量？
5. 词向量的相似性是否等同于词典意义上的相似性？

---

# 十、提高任务：CBOW 与 Skip-gram 对比

## CBOW 的训练目标

CBOW 使用上下文词预测中心词：

```text
上下文词 → 中心词
```

例如：

```text
[猫, 吃] → 喜欢
```

请尝试修改训练样本构造函数，使其生成：



```python
(context_word_ids, center_word_id)
```



并思考：

- CBOW 的输入是否由一个词变成多个词？
- 多个上下文词的向量如何合并？
- 在小语料和大语料场景下，CBOW 与 Skip-gram 可能有哪些差异？

## 负采样与层次 Softmax

本实验使用完整 Softmax。请查阅课堂笔记，说明：

- 为什么大词表下完整 Softmax 计算成本高；
- 负采样如何将多分类问题转化为多个二分类问题；
- 层次 Softmax 为什么可以降低输出计算复杂度；
- 哈夫曼树为什么可能比普通平衡二叉树更适合表示高频词和低频词。

---

# 十一、实验结果与报告要求

## 1. 必须提交的内容

1. 完成后的 Notebook：

```text
NLP实验2_词向量_学号_姓名.ipynb
```

2. 实验报告 PDF；
3. 训练损失曲线；
4. 至少 3 组词语相似度结果；
5. 词向量相似词查询结果；
6. 任务5中的比较分析；
7. 遇到的问题及解决办法。

## 2. 实验报告结构

```text
1. 实验目的
2. 实验环境
3. 独热表示与分布式表示对比
4. Skip-gram训练样本构造
5. 模型结构与训练过程
6. 词向量相似度结果
7. 结果分析
8. Word2Vec、GloVe、ELMo比较
9. 问题记录与解决方法
10. 实验总结
```
---

# 十二、实验自查清单

- [ ] 能够解释独热表示为什么不能反映语义相似性
- [ ] 能够解释窗口大小的含义
- [ ] 已成功构造 Skip-gram 样本
- [ ] 已成功定义并训练 Skip-gram 模型
- [ ] 损失曲线总体下降
- [ ] 已提取词向量矩阵
- [ ] 已计算余弦相似度
- [ ] 已查询至少 3 个词的相似词
- [ ] 已比较 Word2Vec、GloVe 和 ELMo
- [ ] 已分析小语料训练的局限性
- [ ] 已保存 Notebook、截图和实验报告

---

# 十三、实验总结

本实验围绕课堂中的词向量内容，完成从离散表示到分布式表示的实践过渡：

```text
独热表示
  ↓
上下文窗口与训练样本
  ↓
Skip-gram 词向量学习
  ↓
余弦相似度与相似词分析
  ↓
Word2Vec、GloVe、ELMo 方法比较
```

实验的核心结论是：词向量不是人工给每个词指定的标签，而是模型根据词语在语料中的上下文分布学习得到的数值表示。由于词向量的质量依赖语料规模、语料领域、窗口大小和训练方法，因此实验结果必须结合训练条件进行解释。
